In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from MyLib import GPSConverter, es_calc
from scipy.io import netcdf_file as ncf
from scipy.interpolate import griddata
from osgeo import gdal
import matplotlib.path as mplPath
import glob
from netCDF4 import Dataset # pylint: disable=no-name-in-module
import imageio
import cmocean
import tarfile
from scipy.io import netcdf
from xmitgcm import open_mdsdataset as mitgcmds
from matplotlib.path import Path
import time
from datetime import datetime
from datetime import datetime as dt
import matplotlib.dates as md
coord_conv = GPSConverter()

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# # data_path = 'F:\\Datalakes50_diag\\test_results_diag_full\\'
# data_path = 'F:\\mitgcm_HR50_results\\'
# # solnFolders = sorted(glob.glob(f'{data_path}Hydrodynamics/run/pickup.000*.nc'))
# # solnFolders = sorted(glob.glob(f'{data_path}3Dsnap*.nc'))

# # solnFolders = sorted(glob.glob(f'{data_path}3D*.nc'))
# solnFolders = sorted(glob.glob(f'{data_path}3D*.tar.gz'))

In [ ]:
# # solnFolders = solnFolders[0:-10]
# output_path = 'E:/ALPLakes/algal_bloom_Sep2021/Results/new_run/'

# unzipp tar files

In [ ]:
# for ii in range(len(solnFolders)):
#     ftar = solnFolders[ii]
#     tar = tarfile.open(ftar, "r")
#     tar.extractall(data_path+'/unzipped')
#     tar.close()

In [ ]:
# ii = 0
# ftar = solnFolders[ii]
# tar = tarfile.open(ftar, "r")
# print(tar)
# # soln = tar.getnames()[0]
# member = [m for m in tar.getmembers() if solnFolders[ii][-33:-7] in m.name][0]

# variablesFile = netcdf.netcdf_file(tar.extractfile(member),'r')
# # print(soln)
# # variablesFile = Dataset(soln)
# print(variablesFile.variables.keys())

# # tar.close()

# Read and plot other prameters (vorticity, strain, etc.)

In [ ]:
# gcm_directory = 'G:/ALPLakes/algal_bloom_Sep2021/mitgcm_HR50_results'
# gcm_directory = 'G:/ALPLakes/MITgcm/grid50_64cores_secchi_updated/results_5thpart'
gcm_directory = 'I:/MITgcm/MITgcm_Dave/results_2ndpart'

# mitgcm output filename root
gcm_out_root = "additional_params"

# reference date of MITgcm simulation
gcm_start = "2022-08-01 00:00"

# MITgcm time step in seconds
gcm_dt = 4.0

# grid geometry used in the MITgcm simulation
gcm_geometry = "cartesian"


#----------------------------------
# First, we read the grid only
#grid = mitgcmds(gcm_directory, delta_t=gcm_dt, ref_date=gcm_start,
                  #geometry=gcm_geometry, read_grid=False,
                  #prefix=[gcm_out_root], iters=[], swap_dims=False,endian='=')

grid = mitgcmds(gcm_directory, delta_t=gcm_dt, ref_date=gcm_start,
                 geometry=gcm_geometry, read_grid=True,
                 prefix=[gcm_out_root], iters=[], swap_dims=False,endian='=')

#----------------------------------
data = mitgcmds(gcm_directory, delta_t=gcm_dt, ref_date=gcm_start,
                  geometry=gcm_geometry, read_grid=False,
                  prefix=[gcm_out_root],levels=[1.,2.,3.,4.,5.,6.,7.,8.,9.,10.],endian='=')

In [ ]:
np.array(data.momVort3[10,2,:,700])

In [ ]:
np.sum(np.array(grid.hFacC)!=0)/232000

In [ ]:
grid.hFacC.shape

In [ ]:
580000000/50/50

In [ ]:
np.array(grid.hFacC.data.min())

In [ ]:
data

In [ ]:
date_obs = data.time.to_masked_array().filled().astype("datetime64[m]")

In [ ]:
data.momVort3.shape

In [ ]:
dum_arr = data.Strain[:, :, :,:]
momvort_surf = dum_arr.where(dum_arr!=0)

# wvel_surf_avg = (wvel_surf1[:,0:6, :,:]).mean(axis=1,skipna=True)
momvort_surf_avg = (momvort_surf[:,:, :,:]).mean(axis=1,skipna=True)

date_start = np.datetime64(dt.strptime('2021-08-30 08:00',"%Y-%m-%d %H:%M"))
date_end = np.datetime64(dt.strptime('2021-09-03 08:00',"%Y-%m-%d %H:%M"))

idx_start = np.where(abs(date_obs-date_start)==np.min(abs(date_obs-date_start)))[0][0]
idx_end = np.where(abs(date_obs-date_end)==np.min(abs(date_obs-date_end)))[0][0]

momvort_surf_time_avg = momvort_surf_avg[idx_start:idx_end,:,:].mean(axis=0,skipna=True)

In [ ]:
output_path = 'G:/ALPLakes/algal_bloom_Sep2021/new_results/'


colormap = 'seismic'#cmocean.cm.oxy #cmocean.cm.thermal
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for ii in [222]: #range(len(date_obs)):
#     Temp = np.array(wvel_surf_time_avg)*100
    Temp = momvort_surf_time_avg.data*1000
#     Temp[Temp <= 0.005] = np.nan
    #Temp /= 1.05e-4


    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=colormap,vmin=-1e-1, vmax=1e-1)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=16.7, vmax=20.6,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='both',format='%:6.6f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=4),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Horizontal \ Divergence \ [s-1]*1000}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

    #plt.savefig(output_path+'strain_30Aug8to3Sep8.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
np.min(np.min(np.array(data.Strain[i, 0:1, :,:].mean(axis=0))))

In [ ]:
date_obs[185]

In [ ]:
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [185]:
    # Temp = np.array(data.THETA[i, 0:3, :,:].mean(axis=0))
    # Temp[Temp <= 0] = np.nan
    # vort = np.array(data.momVort3[i, 0:1, :,:].mean(axis=0))
    # vort /= 1.05e-4
    vort = np.array(data.momHDiv[i, 0:1, :,:].mean(axis=0))
    #####################
    fig_size = (16,5)


    f = plt.figure(figsize=fig_size)

#     ax = f.add_subplot(121)
#     # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),2), vmax=np.nanpercentile(Temp.flatten(),98),shading='flat')
#     plt.hold=True
#     ax.plot(xs,ys, 'k-', lw=1.5)

#     plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
#     plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

#     ax.set_xlim(497, 565)
#     ax.set_ylim(115, 155)
#     ax.set_aspect('equal')
#     ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


#     cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')

    ax2 = f.add_subplot(122)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,vort, vmin=-2e-4, vmax=2e-4, cmap=cmocean.cm.balance,shading='flat')
    plt.hold=True
    ax2.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax2.set_xlim(497, 565)
    ax2.set_ylim(115, 155)
    ax2.set_aspect('equal')
    ax2.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax2.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    # ax.annotate(date_obs[0].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='both',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Ro=\ (\omega_z /f) \ [-]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

    # plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftemp_vort_2_98perc.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

# read the temperature results

In [ ]:
# gcm_directory = 'G:/ALPLakes/algal_bloom_Sep2021/mitgcm_HR50_results'
# gcm_directory = 'G:/ALPLakes/MITgcm/grid50_secchi_updated_lowest/results_3rdpart'
# gcm_directory = 'G:/ALPLakes/MITgcm/grid50_secchi_updated_lowest/results_test'
gcm_directory = 'I:/MITgcm/MITgcm_Dave/results_3rdpart'
# gcm_directory = 'G:/ALPLakes/MITgcm/grid50_64cores_secchi_updated/results_5thpart'
# gcm_directory = 'G:/ALPLakes/D3D/lac_de_joux/LDJ_10days_2020/mitgcm_format/MITgcm_run/results_from_Wentao'

# mitgcm output filename root
gcm_out_root = "3Dsnaps"

# reference date of MITgcm simulation
gcm_start = "2022-08-01 00:00"

# MITgcm time step in seconds
gcm_dt = 4.0

# grid geometry used in the MITgcm simulation
gcm_geometry = "cartesian"


In [ ]:
# First, we read the grid only
grid = mitgcmds(gcm_directory, delta_t=gcm_dt, ref_date=gcm_start,
                 geometry=gcm_geometry, read_grid=True,
                 prefix=[gcm_out_root], iters=[], swap_dims=False,endian='=')

In [ ]:
np.array(grid.Zl)

In [ ]:
data = mitgcmds(gcm_directory, delta_t=gcm_dt, ref_date=gcm_start,
                 geometry=gcm_geometry, read_grid=False,
                 prefix=[gcm_out_root],endian='=')

In [ ]:
np.array(data.UVEL[1,30,:,:])

In [ ]:
date_obs = data.time.to_masked_array().filled().astype("datetime64[m]")

In [ ]:
date_obs

In [ ]:
np.array(data.iter[:])[31]

In [ ]:
date_obs[-1].item().strftime("%Y-%m-%d %H:%M")

In [ ]:
np.nanmin(np.array(data.WVEL[0, 0:100, :,:].mean(axis=0)))

In [ ]:
(np.array(data.THETA[0,0,:])).shape

In [ ]:
np.array(grid.XG[0, :]).shape

In [ ]:
np.array(grid.YG[:, 0]).shape

In [ ]:
(np.array(data.momVort3[0,0,:])).shape

# plot the results

In [ ]:
x_com = np.array(grid.XG[0, :])
y_com = np.array(grid.YG[:, 0])

xx_com,yy_com = np.meshgrid(x_com,y_com)
(X0, Y0) = (500000, 116500)
(X1, Y1) = (563000, 138700)
gridAngle = np.arctan2(Y1-Y0, X1-X0)
xx_sg = (np.cos(gridAngle)*xx_com - np.sin(gridAngle)*yy_com) + X0
yy_sg = (np.sin(gridAngle)*xx_com + np.cos(gridAngle)*yy_com) + Y0

xx_sg *= 1.e-3
yy_sg *= 1.e-3

In [ ]:
xx_sg

In [ ]:
output_path = 'I:/MITgcm/MITgcm_Dave/plot_results/'
xs, ys = np.genfromtxt("E:/ALPLakes/algal_bloom_Sep2021/Codes/geometery/shoreline_Leman.ldb",unpack=True)
xs *= 1.e-3
ys *= 1.e-3

In [ ]:
poly_path=Path([(xs[ii],ys[ii]) for ii in range(len(xs))])
coors=np.hstack((xx_sg.reshape(-1, 1), yy_sg.reshape(-1,1)))
mask = poly_path.contains_points(coors).reshape(xx_sg.shape)

In [ ]:
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [-10]:#range(len(date_obs)-24,len(date_obs)):
#     Temp = np.array(data.THETA[i, 8:9, :,:].mean(axis=0))
    Temp = np.array(data.THETA[i, 0, :,:])
    Temp[Temp <= 0] = np.nan
    # Temp = Temp-np.nanmean(Temp)
    # vort = np.array(data.momVort3[i, 0:1, :,:].mean(axis=0))
    # vort /= 1.05e-4

    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=np.nanpercentile(Temp.flatten(),0.5), vmax=np.nanmax(Temp.flatten()),shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=20, vmax=24,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

#     ax2 = f.add_subplot(122)
#     # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,vort, vmin=-1.2, vmax=1.2,cmap=cmocean.cm.balance,shading='flat')
#     plt.hold=True
#     ax2.plot(xs,ys, 'k-', lw=1.5)

#     plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
#     plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

#     ax2.set_xlim(497, 565)
#     ax2.set_ylim(115, 155)
#     ax2.set_aspect('equal')
#     ax2.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax2.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     # ax.annotate(date_obs[0].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


#     cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='both',format='%:.0f');
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Ro=\ (\omega_z /f) \ [-]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')

    # plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftemp_vort_2_98perc.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
np.array(data.THETA[1, :, :,:]).mean()

In [ ]:
x_com

In [ ]:
# x_com = np.array(grid.XG[0, :])
# y_com = np.array(grid.YG[:, 0])

# xx_sg,yy_sg = np.meshgrid(x_com,y_com)

# xx_sg *= 1.e-3
# yy_sg *= 1.e-3


x_com = np.array(grid.XG[0, :])
y_com = np.array(grid.YG[:, 0])

xx_com,yy_com = np.meshgrid(x_com,y_com)
(X0, Y0) = (500000, 116500)
(X1, Y1) = (563000, 138700)
gridAngle = np.arctan2(Y1-Y0, X1-X0)
xx_sg = (np.cos(gridAngle)*xx_com - np.sin(gridAngle)*yy_com) + X0
yy_sg = (np.sin(gridAngle)*xx_com + np.cos(gridAngle)*yy_com) + Y0

xx_sg *= 1.e-3
yy_sg *= 1.e-3


font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [3]:#range(len(date_obs)-24,len(date_obs)):
#     Temp = np.array(data.THETA[i, 8:9, :,:].mean(axis=0))
    Temp = np.array(data.VVEL[i, 43, :,:])
    Temp[Temp <= -2] = np.nan
    # Temp = Temp-np.nanmean(Temp)
    # vort = np.array(data.momVort3[i, 0:1, :,:].mean(axis=0))
    # vort /= 1.05e-4

    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=np.nanpercentile(Temp.flatten(),0.5), vmax=np.nanmax(Temp.flatten()),shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=20, vmax=24,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    
    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])
    
#     ax.set_xlim(505, 519)
#     ax.set_ylim(161.5, 164)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar();
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')

#     ax2 = f.add_subplot(122)
#     # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,vort, vmin=-1.2, vmax=1.2,cmap=cmocean.cm.balance,shading='flat')
#     plt.hold=True
#     ax2.plot(xs,ys, 'k-', lw=1.5)

#     plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
#     plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

#     ax2.set_xlim(497, 565)
#     ax2.set_ylim(115, 155)
#     ax2.set_aspect('equal')
#     ax2.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax2.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     # ax.annotate(date_obs[0].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


#     cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='both',format='%:.0f');
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Ro=\ (\omega_z /f) \ [-]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')

    # plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftemp_vort_2_98perc.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
Temp.shape

In [ ]:
xx_sg.shape

In [ ]:
np.nanmax(Temp.flatten())

In [ ]:
date_obs[245].item().strftime("%Y-%m-%d %H:%M")

In [ ]:
output_path = 'I:/MITgcm/MITgcm_Dave/plot_results/'

font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]
skip = 15
temp_range = [-1.1,1.1]

for i in range(len(date_obs)):
    Temp = np.array(data.THETA[i, 0:6, :,:].mean(axis=0))
    Temp[Temp <= 0] = np.nan
    Temp = Temp-np.nanmean(Temp)
#     Temp = Temp-np.nanmean(Temp)
    # vort = np.array(data.momVort3[i, 0:1, :,:].mean(axis=0))
    # vort /= 1.05e-4
    
    uspeed = np.sqrt((np.array(data.UVEL[i, 0:6, :,:])**2)+(np.array(data.VVEL[i, 0:6, :,:])**2)).mean(axis=0)
    uu = np.array(data.UVEL[i, 0:6, :,:].mean(axis=0))/uspeed 
    vv = np.array(data.VVEL[i, 0:6, :,:].mean(axis=0))/uspeed

    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(121)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),1), vmax=np.nanpercentile(Temp.flatten(),99),shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=temp_range[0], vmax=temp_range[1],shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-0.9, vmax=0.9,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=-2.5, vmax=1.5,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])

    cbar = plt.colorbar(fraction=0.02,orientation="horizontal",pad=-0.12,extend='both',format='%:.0f');
    cbar.ax.set_xticklabels([-1,-0.5,0,0.5,1],fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Surface \ temperature \ anomaly \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

    
    ax2 = f.add_subplot(122)
    SS = plt.pcolormesh(xx_sg,yy_sg,np.ma.array(uspeed*100,mask=~mask,fill_value=np.nan), cmap='jet',vmin=0, vmax=30,shading='flat')
    plt.hold=True
    ax2.plot(xs,ys, 'k-', lw=1.5)
    ax2.quiver(xx_sg[::skip, ::skip], yy_sg[::skip, ::skip],uu[::skip, ::skip],vv[::skip, ::skip], color='k',scale=70)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax2.set_xlim(497, 565)
    ax2.set_ylim(115, 155)
    ax2.set_aspect('equal')
    ax2.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax2.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax2.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar2 = plt.colorbar(fraction=0.02,orientation="horizontal",pad=-0.12,extend='max',format='%:.0f');
    cbar2.ax.set_xticklabels(np.array([0,5,10,15,20,25,30]),fontname=font_feature[0],fontsize=font_feature[2])
    cbar2.set_label(label='$\mathregular{Surface \ Current \ velocity \ [cm/s]}$',family=font_feature[0],size=font_feature[1])
    cbar2.ax.xaxis.set_ticks_position('top')

    plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftempanom_current.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
import cv2
import os

image_folder = 'I:/MITgcm/MITgcm_Dave/plot_results/'
video_name = image_folder+'surf_temp_current_0to3m.avi'

images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
frame = cv2.imread(os.path.join(image_folder, images[0]))
height, width, layers = frame.shape

video = cv2.VideoWriter(video_name, 0, 5, (width,height))

for image in images:
    video.write(cv2.imread(os.path.join(image_folder, image)))

cv2.destroyAllWindows()
video.release()

In [ ]:
np.sqrt((np.array(data.UVEL[i, 0:6, :,:])**2)+(np.array(data.VVEL[i, 0:6, :,:])**2)).mean(axis=0).shape

In [ ]:
(data.WVEL[i, 0:1, :,:].mean(axis=0)).shape

In [ ]:
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]
skip = 25
for i in [50]: #range(len(date_obs)):
    uspeed = np.sqrt((np.array(data.UVEL[i, 0:6, :,:])**2)+(np.array(data.VVEL[i, 0:6, :,:])**2)).mean(axis=0)
    uu = np.array(data.UVEL[i, 0:6, :,:].mean(axis=0))/uspeed 
    vv = np.array(data.VVEL[i, 0:6, :,:].mean(axis=0))/uspeed
    # Temp[Temp <= 0] = np.nan
    # vort = np.array(data.momVort3[i, 0:1, :,:].mean(axis=0))
    # vort /= 1.05e-4

    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,np.ma.array(uspeed*100,mask=~mask,fill_value=np.nan), cmap='jet',vmin=0, vmax=25,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)
    ax.quiver(xx_sg[::skip, ::skip], yy_sg[::skip, ::skip],uu[::skip, ::skip],vv[::skip, ::skip], color='k',scale=90)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

#     ax2 = f.add_subplot(122)
#     # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,vort, vmin=-1.2, vmax=1.2,cmap=cmocean.cm.balance,shading='flat')
#     plt.hold=True
#     ax2.plot(xs,ys, 'k-', lw=1.5)

#     plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
#     plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

#     ax2.set_xlim(497, 565)
#     ax2.set_ylim(115, 155)
#     ax2.set_aspect('equal')
#     ax2.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax2.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     # ax.annotate(date_obs[0].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


#     cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='both',format='%:.0f');
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Ro=\ (\omega_z /f) \ [-]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')

    # plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftemp_vort_2_98perc.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
# arr_size = len(date_obs)
# data_median = [np.nan]*arr_size
# data_mean = [np.nan]*arr_size
# data_std = [np.nan]*arr_size
# data_q25 = [np.nan]*arr_size
# data_q75 = [np.nan]*arr_size
# data_q5 = [np.nan]*arr_size
# data_q95 = [np.nan]*arr_size
# datetime_arr = [np.nan]*arr_size
# for ii in range(len(date_obs)):
#     Temp = np.array(data.THETA[ii, 0:6, :,:].mean(axis=0))
#     Temp[Temp <= 0] = np.nan
#     data_median[ii] = np.nanpercentile(Temp,50)
#     data_mean[ii] = np.nanmean(Temp)
#     data_std[ii] = np.nanstd(Temp)
#     data_q5[ii] = np.nanpercentile(Temp,5)
#     data_q25[ii] = np.nanpercentile(Temp,25)
#     data_q75[ii] = np.nanpercentile(Temp,75)
#     data_q95[ii] = np.nanpercentile(Temp,95)
    


In [ ]:
###############
df_cosmo_2021_stat = pd.DataFrame(index=date_obs,columns=['median','std','q5','q25','q75','q95'])
df_cosmo_2021_stat['median'] = data_median
df_cosmo_2021_stat['mean'] = data_mean
df_cosmo_2021_stat['std'] = data_std
df_cosmo_2021_stat['q5'] = data_q5
df_cosmo_2021_stat['q25'] = data_q25
df_cosmo_2021_stat['q75'] = data_q75
df_cosmo_2021_stat['q95'] = data_q95

df_cosmo_2021_stat.to_csv('G:/ALPLakes/algal_bloom_Sep2021/new_results/'+'df_temp_mitgcm_surf_3m_stat.csv')

In [ ]:
input_path = 'G:/ALPLakes/algal_bloom_Sep2021/new_results/'
fname = 'df_temp_mitgcm_surf_3m_stat.csv'
roll_win = 24
df_in = pd.read_csv(input_path+fname,index_col=[0])
df_in_roll = df_in.rolling(roll_win).mean()

date_timestamp = [time.mktime(dt.strptime(ii, '%Y-%m-%d %H:%M:%S').timetuple()) for ii in df_in_roll.index]
date_datetime = [dt.fromtimestamp(ts) for ts in date_timestamp]

In [ ]:
df_in

In [ ]:
y_lim = [17,23.5]
x_lim = [date_datetime[0],date_datetime[-1]]
fig_size = (10,4)
# xy_lbl = ['$\mathregular{Global \ radiation \ {[W/m^{2}]}}$'] # [xlaebl, ylabel]
# xy_lbl = ['$\mathregular{Wind \ speed \ {[m/s]}}$'] # [xlaebl, ylabel]
xy_lbl = ['$\mathregular{Surface \ Temperature \ {[^o{C}]}}$'] # [xlaebl, ylabel]
xtick_rotation = 30
font_feature = ['sans-serif', 16, 16] # [fontname, fontsize_labels, fontsize_ticks]
xtick_dateformat = '%d-%m' # None or e.g., '%d-%m %H'
plt.rcParams['mathtext.fontset'] = 'cm'

#######################
fig = plt.figure(figsize=fig_size)
ax = fig.add_subplot(1,1,1)

plt.grid(axis='x',color='#EBEAED', linestyle='-', linewidth=1.5)
plt.hold = True
ax.plot(date_datetime,df_in['median'],'-',color='k',label='Hourly data')
# ax.plot(date_datetime,df_in_roll['median'],'-',color='k',lw=2,label='1-day moving-averaged')
ax.fill_between(date_datetime, df_in['q75'], df_in['q25'], facecolor='k', alpha=0.3)
# ax.fill_between(date_datetime, df_in_roll['q75'], df_in_roll['q25'], facecolor='#AEAEAE', alpha=0.3)

if xtick_dateformat !=None:
    axd=plt.gca()
    xfmt = md.DateFormatter(xtick_dateformat)
    axd.xaxis.set_major_formatter(xfmt)

ax.set_xlim(xmin=x_lim[0], xmax=x_lim[1])
ax.set_ylim(ymin=y_lim[0], ymax=y_lim[1])
ax.set_xlabel('Time', fontsize= font_feature[1],fontname=font_feature[0])
ax.set_ylabel(xy_lbl[0], fontsize= font_feature[1],fontname=font_feature[0])
if xtick_rotation != 0:
    plt.xticks(rotation=xtick_rotation, fontname=font_feature[0],fontsize=font_feature[2])
else:
    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])
ax.tick_params(axis='y')


# ax.legend(loc='upper right',ncol=3,facecolor = 'k',frameon=False, prop={"size":font_feature[1]-4,'family':font_feature[0]})

plt.savefig(input_path+'SurfTemp_water_3m_stats.png',dpi=300, bbox_inches = 'tight')

plt.show()
plt.close()

In [ ]:
input_path

In [ ]:
y_lim = [17,23.5]
x_lim = [date_datetime[0],date_datetime[-1]]
fig_size = (10,4)
# xy_lbl = ['$\mathregular{Global \ radiation \ {[W/m^{2}]}}$'] # [xlaebl, ylabel]
# xy_lbl = ['$\mathregular{Wind \ speed \ {[m/s]}}$'] # [xlaebl, ylabel]
xy_lbl = ['$\mathregular{Surface \ Temperature \ {[^o{C}]}}$'] # [xlaebl, ylabel]
xtick_rotation = 30
font_feature = ['sans-serif', 16, 16] # [fontname, fontsize_labels, fontsize_ticks]
xtick_dateformat = '%d-%m' # None or e.g., '%d-%m %H'
plt.rcParams['mathtext.fontset'] = 'cm'

#######################
for i in range(len(date_obs)):
    fig = plt.figure(figsize=fig_size)
    ax = fig.add_subplot(1,1,1)

    plt.grid(axis='x',color='#EBEAED', linestyle='-', linewidth=1.5)
    plt.hold = True
    ax.plot(date_datetime,df_in['median'],'-',color='k',label='Hourly data')
    # ax.plot(date_datetime,df_in_roll['median'],'-',color='k',lw=2,label='1-day moving-averaged')
    ax.fill_between(date_datetime, df_in['q75'], df_in['q25'], facecolor='k', alpha=0.3)
    # ax.fill_between(date_datetime, df_in_roll['q75'], df_in_roll['q25'], facecolor='#AEAEAE', alpha=0.3)
    plt.plot([date_datetime[i],date_datetime[i]],y_lim,'--',color='r', linewidth=1)
    if xtick_dateformat !=None:
        axd=plt.gca()
        xfmt = md.DateFormatter(xtick_dateformat)
        axd.xaxis.set_major_formatter(xfmt)

    ax.set_xlim(xmin=x_lim[0], xmax=x_lim[1])
    ax.set_ylim(ymin=y_lim[0], ymax=y_lim[1])
    ax.set_xlabel('Time', fontsize= font_feature[1],fontname=font_feature[0])
    ax.set_ylabel(xy_lbl[0], fontsize= font_feature[1],fontname=font_feature[0])
    if xtick_rotation != 0:
        plt.xticks(rotation=xtick_rotation, fontname=font_feature[0],fontsize=font_feature[2])
    else:
        plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])
    ax.tick_params(axis='y')
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.7, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1],color='r')

    # ax.legend(loc='upper right',ncol=3,facecolor = 'k',frameon=False, prop={"size":font_feature[1]-4,'family':font_feature[0]})

    # plt.savefig(input_path+'new_MITgcm_surftemp_median/'+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftempanom_current.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]
skip = 15

for i in range(len(date_obs)):
    Temp = np.array(data.THETA[i, 0:1, :,:].mean(axis=0))
    Temp[Temp <= 0] = np.nan
    Temp = Temp-np.nanmean(Temp)
    # vort = np.array(data.momVort3[i, 0:1, :,:].mean(axis=0))
    # vort /= 1.05e-4
    
    uspeed = np.sqrt((np.array(data.UVEL[i, 0:6, :,:])**2)+(np.array(data.VVEL[i, 0:6, :,:])**2)).mean(axis=0)
    uu = np.array(data.UVEL[i, 0:6, :,:].mean(axis=0))/uspeed 
    vv = np.array(data.VVEL[i, 0:6, :,:].mean(axis=0))/uspeed

    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(121)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),2), vmax=np.nanpercentile(Temp.flatten(),98),shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-0.9, vmax=0.9,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=-2.5, vmax=1.5,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])

    cbar = plt.colorbar(fraction=0.02,orientation="horizontal",pad=-0.12,extend='both',format='%:.0f');
    cbar.ax.set_xticklabels([-05.,0,0.5],fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Surface \ temperature \ anomaly \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

    
    ax2 = f.add_subplot(122)
    SS = plt.pcolormesh(xx_sg,yy_sg,np.ma.array(uspeed*100,mask=~mask,fill_value=np.nan), cmap='jet',vmin=0, vmax=25,shading='flat')
    plt.hold=True
    ax2.plot(xs,ys, 'k-', lw=1.5)
    ax2.quiver(xx_sg[::skip, ::skip], yy_sg[::skip, ::skip],uu[::skip, ::skip],vv[::skip, ::skip], color='k',scale=70)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax2.set_xlim(497, 565)
    ax2.set_ylim(115, 155)
    ax2.set_aspect('equal')
    ax2.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax2.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax2.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar2 = plt.colorbar(fraction=0.02,orientation="horizontal",pad=-0.12,extend='max',format='%:.0f');
    cbar2.ax.set_xticklabels(np.array([0,5,10,15,20,25]),fontname=font_feature[0],fontsize=font_feature[2])
    cbar2.set_label(label='$\mathregular{Surface \ Current \ velocity \ [cm/s]}$',family=font_feature[0],size=font_feature[1])
    cbar2.ax.xaxis.set_ticks_position('top')

    plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftempanom_current.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
output_path

# Calculate the mixing-related parameters

In [ ]:
xx_sg.shape

In [ ]:
ind_y = np.where(np.array(data.THETA[0,0, :,:])>0)[0]
ind_x = np.where(np.array(data.THETA[0,0, :,:])>0)[1]

In [ ]:
np.array(grid.Zl)

In [ ]:
wvel_surf = data.WVEL[:, :, :,:]
wvel_surf1 = wvel_surf.where(wvel_surf!=0)

# wvel_surf_avg = (wvel_surf1[:,0:6, :,:]).mean(axis=1,skipna=True)
wvel_surf_avg = (wvel_surf1[:,0:16, :,:]).mean(axis=1,skipna=True)

In [ ]:
wvel_surf_avg[10:30,:,:].mean(axis=0,skipna=True).shape

In [ ]:
# date_obs_str = np.array([jj.item().strftime("%Y-%m-%d %H:%M") for jj in date_obs])

date_start = np.datetime64(dt.strptime('2021-08-26 08:00',"%Y-%m-%d %H:%M"))
date_end = np.datetime64(dt.strptime('2021-09-02 08:00',"%Y-%m-%d %H:%M"))

idx_start = np.where(abs(date_obs-date_start)==np.min(abs(date_obs-date_start)))[0][0]
idx_end = np.where(abs(date_obs-date_end)==np.min(abs(date_obs-date_end)))[0][0]

wvel_surf_time_avg = wvel_surf_avg[idx_start:idx_end,:,:].mean(axis=0,skipna=True)

In [ ]:
# np.array(wvel_surf_time_avg.data)

In [ ]:
# np.datetime64(date_start)

In [ ]:
wvel_surf_time_avg.data*100

In [ ]:
output_path = 'G:/ALPLakes/algal_bloom_Sep2021/new_results/'

colormap = 'seismic'#cmocean.cm.oxy #cmocean.cm.thermal
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [222]: #range(len(date_obs)):
#     Temp = np.array(wvel_surf_time_avg)*100
    Temp = wvel_surf_time_avg.data*1000
#     Temp[Temp <= 0.005] = np.nan


    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=colormap,vmin=-0.1, vmax=0.1)
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),1), vmax=np.nanpercentile(Temp.flatten(),99),shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=16.7, vmax=20.6,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=4),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Vertical \ velocity \ [mm/s]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

#     plt.savefig(output_path+'vert_vel_30Aug8to3Sep8.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
colormap = cmocean.cm.oxy_r #cmocean.cm.thermal
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [222]: #range(len(date_obs)):
#     Temp = np.array(wvel_surf_time_avg)*100
    Temp = wvel_surf_time_avg.data*1000
#     Temp[Temp <= 0.005] = np.nan


    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=colormap,vmin=-0.1, vmax=0.1)
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),1), vmax=np.nanpercentile(Temp.flatten(),99),shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=16.7, vmax=20.6,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=4),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Vertical \ velocity \ [mm/s]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

#     plt.savefig(output_path+'vert_vel_30Aug8to3Sep8_ox_cmap.png',dpi=300, bbox_inches = 'tight')
    plt.show()
    plt.close()

In [ ]:
# date_obs_str = np.array([jj.item().strftime("%Y-%m-%d %H:%M") for jj in date_obs])

date_start = np.datetime64(dt.strptime('2021-09-03 00:00',"%Y-%m-%d %H:%M"))
date_end = np.datetime64(dt.strptime('2021-09-06 12:00',"%Y-%m-%d %H:%M"))

idx_start = np.where(abs(date_obs-date_start)==np.min(abs(date_obs-date_start)))[0][0]
idx_end = np.where(abs(date_obs-date_end)==np.min(abs(date_obs-date_end)))[0][0]

wvel_surf_time_avg = wvel_surf_avg[idx_start:idx_end,:,:].mean(axis=0,skipna=True)

In [ ]:
colormap = 'seismic'#cmocean.cm.oxy #cmocean.cm.thermal
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [222]: #range(len(date_obs)):
#     Temp = np.array(wvel_surf_time_avg)*100
    Temp = wvel_surf_time_avg.data*100
#     Temp[Temp <= 0.005] = np.nan


    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=colormap,vmin=-0.01, vmax=0.01)
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),1), vmax=np.nanpercentile(Temp.flatten(),99),shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=16.7, vmax=20.6,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=4),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

    # plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftemp_vort_2_98perc.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
colormap = cmocean.cm.oxy #cmocean.cm.thermal
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

for i in [222]: #range(len(date_obs)):
#     Temp = np.array(wvel_surf_time_avg)*100
    Temp = wvel_surf_time_avg.data*100
#     Temp[Temp <= 0.005] = np.nan


    #####################
    fig_size = (16,10)


    f = plt.figure(figsize=fig_size)

    ax = f.add_subplot(111)
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=colormap,vmin=-0.01, vmax=0.01)
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,vmin=np.nanpercentile(Temp.flatten(),1), vmax=np.nanpercentile(Temp.flatten(),99),shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='jet',vmin=16.7, vmax=20.6,shading='flat')
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap='coolwarm',vmin=-1, vmax=1,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    ax.set_xlim(497, 565)
    ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.annotate(date_obs[i].item().strftime("%Y-%m-%d %H:%M"),xy=(0.02, 0.9), xycoords='axes fraction',fontname=font_feature[0],fontsize=font_feature[1])


    cbar = plt.colorbar(fraction=0.04,orientation="horizontal",pad=-0.25,extend='max',format='%:.0f');
    cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=4),fontname=font_feature[0],fontsize=font_feature[2])
    cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
    cbar.ax.xaxis.set_ticks_position('top')

    # plt.savefig(output_path+date_obs[i].item().strftime("%Y-%m-%d")+'H'+date_obs[i].item().strftime("%H")+'M'+date_obs[i].item().strftime("%M")+'_Surftemp_vort_2_98perc.png',dpi=300, bbox_inches = 'tight')

    plt.show()
    plt.close()

In [ ]:
# # pd.DataFrame(index=range(len(ind_x)),columns=date_obs).astype(float)
# ii = 186
# jj = 100 
# wvel = np.array(data.WVEL[ii, :, ind_y[jj],ind_x[jj]])
# ind_z = np.min([np.where(wvel==0)[0][0],7])


In [ ]:
jj=2000
dum_arr = (data.WVEL[10, :, ind_y[jj],ind_x[jj]])
dum_arr1 = dum_arr.where(dum_arr!=0)

In [ ]:
np.array(dum_arr1)

In [ ]:
np.array(data.WVEL[:, 0:5, ind_y[jj],ind_x[jj]].mean(axis=1)[0:50])

In [ ]:
np.array(data.WVEL[:, 0:5, ind_y[jj],ind_x[jj]].mean(axis=1))

In [ ]:
wvel

In [ ]:
wvel[0:ind_z].mean()

In [ ]:
wvel

In [ ]:

# wvel_all[wvel_all==0] = np.nan                              

In [ ]:
wvel_all[0,0,ind_y,ind_y]

In [ ]:
wvel_all[0,0,wvel_all[0,0,:,:]==0]

In [ ]:
np.array(data.WVEL[ii, :, ind_y[jj],ind_x[jj]])

In [ ]:
ind_z

In [ ]:
wvel[0:ind_z].mean()

In [ ]:
len(date_obs)

In [ ]:
wvel

In [ ]:
np.array(data.THETA[ii, :, jj,qq])

In [ ]:
np.array(grid.Zl)

In [ ]:
len(ind_x)

In [ ]:
np.array(data.WVEL[ii, :, :,:]).shape

In [ ]:
output_path

# Make the movie

In [ ]:
import cv2
import os

image_folder = 'G:/ALPLakes/algal_bloom_Sep2021/Particle_tracking/PT_secchi_low/PT9_backward_east/pt_probability_to3/'
video_name = image_folder+'PT_east_backward.avi'

images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
frame = cv2.imread(os.path.join(image_folder, images[0]))
height, width, layers = frame.shape

video = cv2.VideoWriter(video_name, 0, 5, (width,height))

for image in images:
    video.write(cv2.imread(os.path.join(image_folder, image)))

cv2.destroyAllWindows()
video.release()

In [ ]:
import cv2
import os

image_folder = 'G:/ALPLakes/algal_bloom_Sep2021/Particle_tracking/PT2_backward_center/pt_probability/'
video_name = 'G:/ALPLakes/algal_bloom_Sep2021/Particle_tracking/PT2_backward_center/'+'pt_probab_PT2.avi'

images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
frame = cv2.imread(os.path.join(image_folder, images[0]))
height, width, layers = frame.shape

video = cv2.VideoWriter(video_name, 0, 1, (width,height))

for image in images:
    video.write(cv2.imread(os.path.join(image_folder, image)))

cv2.destroyAllWindows()
video.release()

In [ ]:
import moviepy.video.io.ImageSequenceClip

# plot the results

In [ ]:
data_path = 'F:/Datalakes50_new_results_fourthdpart_updated_cosmo/unzipped/'
# solnFolders = sorted(glob.glob(f'{data_path}Hydrodynamics/run/pickup.000*.nc'))
# solnFolders = sorted(glob.glob(f'{data_path}3Dsnap*.nc'))

solnFolders = sorted(glob.glob(f'{data_path}3D*.nc'))

In [ ]:
# min_arr = []
# max_arr= []
# for ii in range(8):
#     min_arr.append(np.min(variablesFile.variables['THETA'][ii,2,:,:]))
#     max_arr.append(np.max(variablesFile.variables['THETA'][ii,2,:,:]))
# plt.plot(min_arr)
# plt.hold = True
# plt.plot(max_arr)

In [ ]:
xs, ys = np.genfromtxt("E:/ALPLakes/algal_bloom_Sep2021/Codes/geometery/shoreline_Leman.ldb",unpack=True)
xs *= 1.e-3
ys *= 1.e-3



#####################
fig_size = (14,8)
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

f = plt.figure(figsize=fig_size)

for ii in range(len(solnFolders)):
    soln = solnFolders[ii]
    variablesFile = Dataset(soln)
    
    x_com = variablesFile.variables['X']
    y_com = variablesFile.variables['Y']

    xx_com,yy_com = np.meshgrid(x_com,y_com)
    (X0, Y0) = (500000, 116500)
    (X1, Y1) = (563000, 138700)
    gridAngle = np.arctan2(Y1-Y0, X1-X0)
    xx_sg = (np.cos(gridAngle)*xx_com - np.sin(gridAngle)*yy_com) + X0
    yy_sg = (np.sin(gridAngle)*xx_com + np.cos(gridAngle)*yy_com) + Y0
    xx_sg *= 1.e-3
    yy_sg *= 1.e-3
    
    
    Temp = variablesFile.variables['THETA'][177, 0, :,:]
    Temp[Temp <= 0] = np.nan

    ax = f.add_subplot(111)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp,vmin=17.8, vmax=19.8, cmap=cmocean.cm.thermal,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    # ax.set_xlim(497, 565)
    # ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    
#     cbar = plt.colorbar(fraction=0.05,orientation="horizontal",pad=-0.3,extend='max',format='%:.0f');
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')
# plt.savefig(output_path+'surf_temp_06Sep_updated_cosmo.png',dpi=300, bbox_inches = 'tight')

plt.show()
plt.close()

In [ ]:
# xs, ys = np.genfromtxt("E:/ALPLakes/algal_bloom_Sep2021/Codes/geometery/shoreline_Leman.ldb",unpack=True)
# xs *= 1.e-3
# ys *= 1.e-3



# #####################
# fig_size = (14,8)
# font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

# f = plt.figure(figsize=fig_size)

# for ii in range(len(solnFolders)):
#     ftar = solnFolders[ii]
#     tar = tarfile.open(ftar, "r")
# #     soln = solnFolders[ii]
#     member = [m for m in tar.getmembers() if solnFolders[ii][-33:-7] in m.name][0]
#     variablesFile = netcdf.netcdf_file(tar.extractfile(member),'r')
    
#     x_com = variablesFile.variables['X'][:]
#     y_com = variablesFile.variables['Y'][:]

#     xx_com,yy_com = np.meshgrid(x_com,y_com)
#     (X0, Y0) = (500000, 116500)
#     (X1, Y1) = (563000, 138700)
#     gridAngle = np.arctan2(Y1-Y0, X1-X0)
#     xx_sg = (np.cos(gridAngle)*xx_com - np.sin(gridAngle)*yy_com) + X0
#     yy_sg = (np.sin(gridAngle)*xx_com + np.cos(gridAngle)*yy_com) + Y0
#     xx_sg *= 1.e-3
#     yy_sg *= 1.e-3
    
    
#     Temp = variablesFile.variables['THETA'][153, 0, :,:]
#     Temp[Temp <= 0] = np.nan

#     ax = f.add_subplot(111)
#     # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
#     SS = plt.pcolormesh(xx_sg,yy_sg,Temp,vmin=17.5, vmax=19.5, cmap=cmocean.cm.thermal,shading='flat')
#     plt.hold=True
#     ax.plot(xs,ys, 'k-', lw=1.5)

#     plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
#     plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

#     # ax.set_xlim(497, 565)
#     # ax.set_ylim(115, 155)
#     ax.set_aspect('equal')
#     ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
#     ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    
#     tar.close()
# #     cbar = plt.colorbar(fraction=0.05,orientation="horizontal",pad=-0.3,extend='max',format='%:.0f');
# #     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
# #     cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
# #     cbar.ax.xaxis.set_ticks_position('top')
# # plt.savefig(output_path+'surf_temp_05Sep.png',dpi=300, bbox_inches = 'tight')

# plt.show()
# plt.close()

In [ ]:
x_com

In [ ]:
xs, ys = np.genfromtxt("E:/ALPLakes/algal_bloom_Sep2021/Codes/geometery/shoreline_Leman.ldb",unpack=True)
xs *= 1.e-3
ys *= 1.e-3



#####################
fig_size = (14,8)
font_feature = ['sans-serif', 16, 14] # [fontname, fontsize_labels, fontsize_ticks]

f = plt.figure(figsize=fig_size)

for ii in range(len(solnFolders)):
    soln = solnFolders[ii]
    variablesFile = Dataset(soln)
    
    x_com = variablesFile.variables['Xp1']
    y_com = variablesFile.variables['Yp1']

    xx_com,yy_com = np.meshgrid(x_com,y_com)
    (X0, Y0) = (500000, 116500)
    (X1, Y1) = (563000, 138700)
    gridAngle = np.arctan2(Y1-Y0, X1-X0)
    xx_sg = (np.cos(gridAngle)*xx_com - np.sin(gridAngle)*yy_com) + X0
    yy_sg = (np.sin(gridAngle)*xx_com + np.cos(gridAngle)*yy_com) + Y0
    xx_sg *= 1.e-3
    yy_sg *= 1.e-3
    
    
    Temp = variablesFile.variables['momVort3'][177, 0:1, :,:].mean(axis=0)
#     Temp[Temp <= 0] = np.nan

    ax = f.add_subplot(111)
    # SS = plt.pcolormesh(xx_sg,yy_sg,Temp, cmap=cmocean.cm.thermal,shading='flat')
    SS = plt.pcolormesh(xx_sg,yy_sg,Temp,vmin=-1e-4, vmax=1e-4,cmap=cmocean.cm.balance,shading='flat')
    plt.hold=True
    ax.plot(xs,ys, 'k-', lw=1.5)

    plt.xticks(fontname=font_feature[0],fontsize=font_feature[2])
    plt.yticks(fontname=font_feature[0],fontsize=font_feature[2])

    # ax.set_xlim(497, 565)
    # ax.set_ylim(115, 155)
    ax.set_aspect('equal')
    ax.set_xlabel("Lat (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])
    ax.set_ylabel("Lon (km CH1903)",fontname=font_feature[0],fontsize=font_feature[1])

#     cbar = plt.colorbar(fraction=0.05,orientation="horizontal",pad=-0.3,extend='max',format='%:.0f');
#     cbar.ax.set_xticklabels(np.around(cbar.ax.get_xticks(), decimals=2),fontname=font_feature[0],fontsize=font_feature[2])
#     cbar.set_label(label='$\mathregular{Surface \ temperature \ [^{o}C]}$',family=font_feature[0],size=font_feature[1])
#     cbar.ax.xaxis.set_ticks_position('top')

# plt.savefig(output_path+'surf_vort_06Sep_updated_cosmo.png',dpi=300, bbox_inches = 'tight')

plt.show()
plt.close()

In [ ]:
variablesFile.variables['momVort3'][177, 0, :,:].shape

In [ ]:
variablesFile.variables['momVort3'][177, 0:10, :,:].mean(axis=0).shape

In [ ]:
ii=1
solnFolders = sorted(glob.glob(f'{data_path}ext*.nc'))

soln = solnFolders[ii]
print(solnFolders[ii])
variablesFile = Dataset(soln)
variablesFile.variables.keys()

In [ ]:
variablesFile.variables['EXFuwind'].shape

In [ ]:
np.max(variablesFile.variables['EXFlwnet'][4,0,:])+np.max(variablesFile.variables['EXFlwnet'][1,0,:])

In [ ]:
np.max(variablesFile.variables['EXFswnet'][1,0,:])+np.max(variablesFile.variables['EXFswnet'][5,0,:])

In [ ]:
min_arr = []
max_arr= []
for ii in range(8):
    min_arr.append(np.min(variablesFile.variables['EXFswnet'][ii,0,:,:]))
    max_arr.append(np.max(variablesFile.variables['EXFswnet'][ii,0,:,:]))
plt.plot(min_arr)
plt.hold = True
plt.plot(max_arr)

In [ ]:
min_arr = []
max_arr= []
for ii in range(24):
    min_arr.append(np.min(variablesFile.variables['EXFpress'][ii,0,:,:]))
    max_arr.append(np.max(variablesFile.variables['EXFpress'][ii,0,:,:]))
plt.plot(min_arr)
plt.hold = True
plt.plot(max_arr)

# Test cosmo2mitgcm

In [ ]:
import numpy as np
from scipy.io.netcdf import netcdf_file
from scipy.interpolate import griddata
from datetime import datetime
from os.path import join
from MyLib import GPSConverter
from pandas import to_datetime

In [ ]:
def get_MITgcm_grid(mitgcm_file):
    #%Extract geographical data from the first COSMO-2 file
    #date = datevec(dateini); %Date vector
    #FileName = [datapath sprintf('%i',date(1)) '\cosmo2_epfl_lakes_' sprintf('%i%02i%02i',date(1:3)) '.nc'];
    f = netcdf_file(mitgcm_file, 'r')
    # Must transpose arrays to be consistent with Fortran ordering
    xC = f.variables["XC"][:].copy().T
    yC = f.variables["YC"][:].copy().T
    f.close()
    return xC, yC

# extract COSMO grid info
def get_COSMO_grid(cosmo_file):
    #%Extract geographical data from the first COSMO-2 file
    #date = datevec(dateini); %Date vector
    #FileName = [datapath sprintf('%i',date(1)) '\cosmo2_epfl_lakes_' sprintf('%i%02i%02i',date(1:3)) '.nc'];
    f = netcdf_file(cosmo_file, 'r')
    lon = f.variables["lon_1"][:].copy()
    lat = f.variables["lat_1"][:].copy()
    f.close()
    converter = GPSConverter()
    x = np.zeros_like(lon)
    y = np.zeros_like(lat)
    for jj in range(x.shape[1]):
        for ii in range(x.shape[0]):
            x[ii,jj], y[ii,jj], _ = converter.WGS84toLV03(
                                    lat[ii, jj], lon[ii,jj], 0.0)
    return x, y

# get time axis info from COSMO file
def get_time(ncf):
    time = ncf.variables["time"][:].copy().astype("int")
    refdate = ncf.variables["time"].units.decode('utf8')
    refdate = refdate.split()
    if refdate[0] == "seconds":
        units = "[s]"
    elif refdate[0] == "hours":
        units = "[h]"
    else:
        raise ValueError("Only seconds are implemented")
    refdate = refdate[2].rstrip(",; ") + " " + refdate[3].rstrip(",; ")
    return np.array([np.datetime64(refdate) + np.timedelta64(t, units)
                    for t in time])

def check_var(v, varrange):
    if (v.min() < varrange[0]) | (v.max() > varrange[1]):
        return True
    else:
        return False

In [ ]:
# configuration
#name = ("atemp",  "aqh", "swdown",     "lwdown", "apressure")
#ncvar = ("T_2M",  "AQH",   "GLOB",   "LW_IN_TG",        "PS")
#shift = (    0.,     0.,       0.,           0.,          0.)
#factor = (   1.,     1.,        1,           1.,          1.)
#name = ("atemp",  "aqh", "swdown",     "lwdown",   "precip", "apressure")
#ncvar = ("T_2M",  "AQH",   "GLOB",   "LW_IN_TG", "TOT_PREC",        "PS")
#shift = (    0.,     0.,       0.,           0.,         0.,          0.)
#factor = (   1.,     1.,        1,           1.,   1./3.6e6,          1.)
name = ("atemp",  "aqh", "swdown",     "lwdown", "apressure")
ncvar = ("T_2M",  "AQH",   "GLOB",   "LW_IN_TG",        "PS")
shift = (    0.,     0.,       0.,           0.,          0.)
factor = (   1.,     1.,        1,           1.,          1.)
vrange = ([230, 320], [1e-5, 0.1], [0, 2e3], [0, 2e3], [5e4, 2e5])

mitgcm_grid_file = "E:/ALPLakes/Codes/MITgcm_input_output/input_generator/LemanGrid_HR50.nc"
out_fmt = ">f4"

shf = dict(zip(name, shift))
fac = dict(zip(name, factor))
var = dict(zip(name, ncvar))
vrn = dict(zip(ncvar, vrange))

In [ ]:
data_path = 'F:/COSMO/2021/'
data_root='cosmo2_epfl_lakes_'
start = '2021-07-26'
end ='2021-07-27'
xinfo = [500000.0,563000.0,1000.0]
yinfo =[116500.0,138700.0,1000.0]
output_path = 'F:/Datalakes50_diag/binary_data/'

In [ ]:
date_start = np.datetime64(start)
date_end = np.datetime64(end)

# Get grid info
fname = data_root + to_datetime(date_start).strftime("%Y%m%d") + ".nc"
fname = join(data_path,fname)

# generate output grid
xM, yM = get_MITgcm_grid(mitgcm_grid_file)

In [ ]:
xM.shape

In [ ]:
date_end

In [ ]:
today = date_start
missing_day = False
date_file = today  
fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
fname = join(data_path, fname)

xc, yc = get_COSMO_grid(fname)
ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
               (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
in_f = netcdf_file(fname, 'r')

print("\ndate: %s\nfile: %s" % (today, fname))

In [ ]:
nc_time = get_time(in_f)
req_time = np.arange(today,
                     today + np.timedelta64(1, 'D'),
                     np.timedelta64(1, 'h'))

In [ ]:
req_time

In [ ]:
fls = {}
for e in name:
    out_name = output_path+"%s_%s_%s.bin" % \
               (e, start.replace('-',''), end.replace('-',''))
    fls[e] = open(out_name, "w")

In [ ]:
for e, f in fls.items():
    v = var[e]
    try:
        if len(in_f.variables[v].shape) == 4:
            data = in_f.variables[v][:, 0, ind[0], ind[1]]
            data *= fac[e]
            data += shf[e]
        elif len(in_f.variables[v].shape) == 3:
            data = in_f.variables[v][:, ind[0], ind[1]]
            data *= fac[e]
            data += shf[e]
        if v == "TOT_PREC":
            data[data < 0] = 0.0
        if v == "PS":
            # we check if some idiot suddenly decided
            # to change the units of pressure
            if np.max(data) < 1e4:
                data *= 100
    except KeyError:
        if v == "AQH":
            relh = in_f.variables["RELHUM_2M"][:, ind[0], ind[1]]
            tmp = in_f.variables["T_2M"][:, ind[0], ind[1]] - 273.15
            ps = in_f.variables["PS"][:, ind[0], ind[1]]
            if np.max(ps) < 1e4:
                ps *= 100
            ew = es_calc(tmp) * 0.01 * relh
            rv = 0.622 * ew / (ps - ew)
            data = rv / (1 + rv)
        else:
            raise KeyError("'%s' variable not found" % v)
    if check_var(data, vrn[v]):
        print("%s range: %.3g - %.3g" % (v, data.min(), data.max()))
    n_miss = 0
    for hh in req_time:
        try:
            index = np.where(nc_time == hh)[0][0]
            out_data = griddata((xc[ind[0], ind[1]], yc[ind[0], ind[1]]),
                                data[index, ...],
                                (xM.ravel(), yM.ravel()),
                                fill_value=np.nan, method="linear")
            bad = np.isnan(out_data)
            # if there are missing points, we fill them with
            # nearest neighbour values
            if np.any(bad > 0):
                out_data[bad] = griddata((xc[ind[0], ind[1]],
                                          yc[ind[0], ind[1]]),
                                          data[index, ...],
                                          (xM.ravel()[bad],
                                           yM.ravel()[bad]),
                                          method="nearest")
            out_data = np.reshape(out_data, xM.shape)
        except IndexError:
            print("Warning: missing data in file!")
            # we do not need to do anything, we will just be writing the
            # last out_data array we computed
            n_miss += 1
        # write to file
        for kk in range(xM.shape[1]):
            out_data[:,kk].astype(out_fmt).tofile(f)
    if n_miss > 0:
        print("Not all expected times are available in this file:\n %s"
              % nc_time)
        if n_miss == 24:
            raise ValueError("Nothing in this file")

in_f.close()
f.close()

In [ ]:
np.min(out_data)

In [ ]:
out_data[:,kk].astype(out_fmt)

In [ ]:
f

In [ ]:
out_fmt = "float"
out_name = output_path+"%s_%s_%s.bin" % \
               ("apressure", start.replace('-',''), end.replace('-',''))
f = open(out_name, "w")

In [ ]:
out_data[:,kk].astype(out_fmt).tofile(f)
f.close()

In [ ]:
input_path = 'F:/Datalakes50_diag/binary_data/'
Q = np.fromfile(input_path+'apressure_20210726_20210727.bin')

In [ ]:
Q

In [ ]:
input_path = 'F:/Datalakes50_diag/binary_data/'
Q = np.fromfile(input_path+'wspeed_20210902_20210905.bin')

In [ ]:
Q.shape

In [ ]:
68124672/528/1344

In [ ]:
# Q_reshpe = Q.reshape(1128,528, 1344)
Q_reshpe = Q.reshape(96,528, 1344)

In [ ]:
plt.imshow(Q_reshpe[78,:,:]) 
plt.show()

In [ ]:
np.mean(Q_reshpe[6,:,:])

In [ ]:
np.max(Q_reshpe[5,:,:])

In [ ]:
Q_reshpe[4,:,:].dtype

In [ ]:
Q_reshpe[3,:,:].dtype